# Tandem Real vs AI Image Classifier

This notebook trains a **dual-branch model** on Hugging Face dataset **`ShreyashDhoot/Ai-vs_Real`**:

- **Deep CNN branch** for visual representation learning.
- **GLCM texture branch** that computes **12 texture features** across **8 directions** (total 96 features).

Both branches are fused exactly at the point where the CNN fully connected head begins.

In [ ]:
# If needed, uncomment this line and run once:
# !pip install -q torch torchvision datasets scikit-image scikit-learn tqdm

import math
import random
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from PIL import Image
from skimage.feature import graycomatrix
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

# -----------------------------
# Reproducibility and device
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Data configuration
# -----------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# GLCM settings: 8 directions
GLCM_LEVELS = 64
GLCM_DISTANCES = [1]
GLCM_ANGLES = np.deg2rad([0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5])
# 12 features x 8 directions = 96 dimensions
GLCM_FEATURE_DIM = 12 * len(GLCM_ANGLES)


# -----------------------------
# Utility transforms (torch only)
# -----------------------------
def pil_to_chw_float(img: Image.Image, size: int = IMG_SIZE) -> torch.Tensor:
    img = img.convert("RGB").resize((size, size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0  # H, W, C in [0, 1]
    arr = np.transpose(arr, (2, 0, 1))              # C, H, W
    return torch.from_numpy(arr)


def normalize_tensor(x: torch.Tensor) -> torch.Tensor:
    # ImageNet-like normalization for stable CNN training
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=x.dtype).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=x.dtype).view(3, 1, 1)
    return (x - mean) / std


def rgb_to_quantized_gray(img: Image.Image, size: int = IMG_SIZE, levels: int = GLCM_LEVELS) -> np.ndarray:
    # Resize, convert to grayscale, and quantize into discrete levels for GLCM
    gray = img.convert("L").resize((size, size), Image.BILINEAR)
    arr = np.asarray(gray, dtype=np.uint8)
    q = (arr.astype(np.float32) / 256.0 * levels).astype(np.int32)
    q = np.clip(q, 0, levels - 1)
    return q


# -----------------------------
# 12 GLCM features per direction
# -----------------------------
def _glcm_features_from_matrix(P: np.ndarray) -> np.ndarray:
    """
    Compute 12 scalar features from one normalized GLCM matrix P.
    Features:
    1) contrast
    2) dissimilarity
    3) homogeneity
    4) ASM
    5) energy
    6) correlation
    7) entropy
    8) mean_i
    9) mean_j
    10) var_i
    11) cluster_shade
    12) cluster_prominence
    """
    eps = 1e-12
    P = P.astype(np.float64)
    P_sum = P.sum()
    if P_sum <= 0:
        return np.zeros(12, dtype=np.float32)
    P = P / (P_sum + eps)

    n = P.shape[0]
    i_idx = np.arange(n).reshape(-1, 1)
    j_idx = np.arange(n).reshape(1, -1)

    diff = i_idx - j_idx
    sum_ij = i_idx + j_idx

    contrast = np.sum((diff ** 2) * P)
    dissimilarity = np.sum(np.abs(diff) * P)
    homogeneity = np.sum(P / (1.0 + (diff ** 2)))

    asm = np.sum(P ** 2)
    energy = math.sqrt(max(asm, 0.0))

    p_i = P.sum(axis=1)
    p_j = P.sum(axis=0)

    mean_i = np.sum(i_idx[:, 0] * p_i)
    mean_j = np.sum(j_idx[0, :] * p_j)

    var_i = np.sum(((i_idx[:, 0] - mean_i) ** 2) * p_i)
    var_j = np.sum(((j_idx[0, :] - mean_j) ** 2) * p_j)

    std_i = math.sqrt(max(var_i, 0.0))
    std_j = math.sqrt(max(var_j, 0.0))

    corr_num = np.sum((i_idx - mean_i) * (j_idx - mean_j) * P)
    correlation = corr_num / (std_i * std_j + eps)

    entropy = -np.sum(P * np.log2(P + eps))

    centered_sum = sum_ij - mean_i - mean_j
    cluster_shade = np.sum((centered_sum ** 3) * P)
    cluster_prominence = np.sum((centered_sum ** 4) * P)

    feats = np.array([
        contrast,
        dissimilarity,
        homogeneity,
        asm,
        energy,
        correlation,
        entropy,
        mean_i,
        mean_j,
        var_i,
        cluster_shade,
        cluster_prominence,
    ], dtype=np.float32)
    return feats


def extract_glcm_96_features(img: Image.Image) -> np.ndarray:
    quant = rgb_to_quantized_gray(img, size=IMG_SIZE, levels=GLCM_LEVELS)
    glcm = graycomatrix(
        quant,
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=GLCM_LEVELS,
        symmetric=True,
        normed=True,
    )

    # glcm shape: (levels, levels, num_distances=1, num_angles=8)
    all_feats: List[np.ndarray] = []
    for a in range(glcm.shape[3]):
        P = glcm[:, :, 0, a]
        all_feats.append(_glcm_features_from_matrix(P))

    feats = np.concatenate(all_feats, axis=0)  # (96,)
    return feats.astype(np.float32)


# -----------------------------
# HF dataset wrapper
# -----------------------------
def infer_label_column(ds_split) -> str:
    # Prefer conventional label keys if available
    for key in ["label", "labels", "target", "class"]:
        if key in ds_split.features:
            return key
    # Fallback: first integer-like column excluding image
    for key, feat in ds_split.features.items():
        if key == "image":
            continue
        # Many datasets expose ClassLabel; casting to int usually works at item level
        if hasattr(feat, "num_classes") or "int" in str(feat):
            return key
    raise ValueError("Could not infer label column. Please set it manually.")


class AIVsRealDataset(Dataset):
    def __init__(self, hf_split, label_col: str, augment: bool = False):
        self.ds = hf_split
        self.label_col = label_col
        self.augment = augment

    def __len__(self):
        return len(self.ds)

    def _maybe_augment(self, img: Image.Image) -> Image.Image:
        if not self.augment:
            return img
        # Light augmentations
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.2:
            img = img.rotate(random.uniform(-8, 8), resample=Image.BILINEAR)
        return img

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ex = self.ds[idx]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))

        img = self._maybe_augment(img)

        cnn_x = pil_to_chw_float(img)
        cnn_x = normalize_tensor(cnn_x)

        glcm_x = extract_glcm_96_features(img)

        y = int(ex[self.label_col])

        return {
            "image": cnn_x,
            "glcm": torch.from_numpy(glcm_x),
            "label": torch.tensor(y, dtype=torch.long),
        }


# -----------------------------
# Model: CNN + GLCM-DNN fusion
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class CNNWithTextureFusion(nn.Module):
    def __init__(self, num_classes: int = 2, glcm_dim: int = GLCM_FEATURE_DIM):
        super().__init__()

        # Deep CNN feature extractor
        self.cnn_backbone = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            nn.Conv2d(256, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Texture feature extractor (DNN branch)
        self.texture_mlp = nn.Sequential(
            nn.Linear(glcm_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
        )

        # Fusion point: where CNN fully connected head starts
        self.classifier = nn.Sequential(
            nn.Linear(512 + 64, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, image: torch.Tensor, glcm: torch.Tensor) -> torch.Tensor:
        cnn_feat = self.cnn_backbone(image)
        cnn_feat = cnn_feat.flatten(1)          # (B, 512)

        tex_feat = self.texture_mlp(glcm)       # (B, 64)

        fused = torch.cat([cnn_feat, tex_feat], dim=1)
        logits = self.classifier(fused)
        return logits


# -----------------------------
# Train / eval helpers
# -----------------------------
@dataclass
class TrainConfig:
    lr: float = 1e-3
    epochs: int = 10
    weight_decay: float = 1e-4


def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    criterion = nn.CrossEntropyLoss()

    pbar = tqdm(loader, desc="Train" if train_mode else "Eval", leave=False)
    for batch in pbar:
        x_img = batch["image"].to(device, non_blocking=True)
        x_glcm = batch["glcm"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(x_img, x_glcm)
            loss = criterion(logits, y)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)
        total_loss += loss.item() * y.size(0)

        pbar.set_postfix(
            loss=f"{(total_loss / max(total_samples, 1)):.4f}",
            acc=f"{(total_correct / max(total_samples, 1)):.4f}",
        )

    avg_loss = total_loss / max(total_samples, 1)
    avg_acc = total_correct / max(total_samples, 1)
    return avg_loss, avg_acc


def predict_all(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            x_img = batch["image"].to(device)
            x_glcm = batch["glcm"].to(device)
            y = batch["label"].to(device)
            logits = model(x_img, x_glcm)
            pred = torch.argmax(logits, dim=1)
            ys.append(y.cpu().numpy())
            ps.append(pred.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


# -----------------------------
# Load HF dataset and build splits
# -----------------------------
raw = load_dataset("ShreyashDhoot/Ai-vs_Real")
print(raw)

# Find label column from first available split
first_split = next(iter(raw.keys()))
label_col = infer_label_column(raw[first_split])
print(f"Detected label column: {label_col}")

# Standardize split availability
if "train" in raw and "validation" in raw and "test" in raw:
    ds_train = raw["train"]
    ds_val = raw["validation"]
    ds_test = raw["test"]
elif "train" in raw and "test" in raw:
    # Create validation from train
    split = raw["train"].train_test_split(test_size=0.1, seed=SEED)
    ds_train = split["train"]
    ds_val = split["test"]
    ds_test = raw["test"]
elif "train" in raw:
    # Create both val and test from train
    split1 = raw["train"].train_test_split(test_size=0.2, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)
    ds_train = split1["train"]
    ds_val = split2["train"]
    ds_test = split2["test"]
else:
    # Fallback: use first split and create train/val/test
    split1 = raw[first_split].train_test_split(test_size=0.2, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)
    ds_train = split1["train"]
    ds_val = split2["train"]
    ds_test = split2["test"]

print(f"Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}")

train_ds = AIVsRealDataset(ds_train, label_col=label_col, augment=True)
val_ds = AIVsRealDataset(ds_val, label_col=label_col, augment=False)
test_ds = AIVsRealDataset(ds_test, label_col=label_col, augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

# Infer number of classes
try:
    feat = ds_train.features[label_col]
    num_classes = getattr(feat, "num_classes", None) or len(set(ds_train[label_col]))
except Exception:
    num_classes = len(set(ds_train[label_col]))

print(f"Num classes: {num_classes}")

# -----------------------------
# Train
# -----------------------------
cfg = TrainConfig(lr=1e-3, epochs=10, weight_decay=1e-4)
model = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

best_val_acc = -1.0
best_state = None

for epoch in range(1, cfg.epochs + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)

    print(
        f"Epoch {epoch:02d}/{cfg.epochs} | "
        f"Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f} | "
        f"Val Loss: {va_loss:.4f}, Val Acc: {va_acc:.4f}"
    )

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

print(f"Best Val Acc: {best_val_acc:.4f}")

# -----------------------------
# Test evaluation
# -----------------------------
y_true, y_pred = predict_all(model, test_loader)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

# Save model
save_path = "ai_vs_real_tandem_cnn_glcm.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved model weights to: {save_path}")